#### So sánh feature CŨ vs MỚI cho model GIÁ CƠ BẢN

So `B_NUM` hiện tại (`_common_train.py`) với bản **thêm 4 feature mới** tính trên chuyến quan sát
gần nhất: tốc độ, phút/km, đơn giá cơ bản/km, giá cơ bản quan sát gần nhất.

Dùng **HistGB** (đã kiểm chứng trước đó: 3 thuật toán boosting cho kết quả gần như nhau — nên chỉ
cần 1 thuật toán để so sánh **feature** có giúp gì không, không cần lặp lại cho cả 3).

Yêu cầu: đã chạy `chuan_bi_du_lieu_v2.ipynb` để có `../../data/hcm_train_ready_v2.parquet`.

**1. Nạp dữ liệu V2 + định nghĩa 2 bộ feature (CŨ vs MỚI)**

In [1]:
import warnings, time
from pathlib import Path
import numpy as np, pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)

CAT = ["service_name", "pickup_location_name", "dropoff_location_name", "weather_main"]

# Bo feature CU (dung y het B_NUM trong _common_train.py)
B_NUM_CU = ["quote_distance", "quote_duration", "gio_vn", "latest_observed_base",
            "history_60m_price_mean", "history_60m_price_std", "history_60m_price_slope_per_minute",
            "latest_observed_quote_distance", "latest_observed_quote_duration", "actual_observation_age_minutes"]

# Bo feature MOI = CU + 3 feature moi (latest_observed_base da co san trong CU roi)
FEATURE_MOI = ["latest_observed_speed_kmh", "latest_observed_dur_per_km", "latest_observed_base_per_km"]
B_NUM_MOI = B_NUM_CU + FEATURE_MOI

PREP = Path("../../data/hcm_train_ready_v2.parquet")
assert PREP.exists(), "Chua co hcm_train_ready_v2.parquet -> chay chuan_bi_du_lieu_v2.ipynb truoc!"
# hcm_train_ready_v2.parquet da tinh san base_price + cac cot dan xuat moi (xem chuan_bi_du_lieu_v2.ipynb)
COLS = list(dict.fromkeys(CAT + B_NUM_MOI + ["base_price", "evaluation_month", "split"]))
df = pd.read_parquet(PREP, columns=COLS)
print(f"Nap {len(df):,} dong | Feature CU: {len(B_NUM_CU)} | Feature MOI: {len(B_NUM_MOI)} (+{len(FEATURE_MOI)})")
print("Them moi:", FEATURE_MOI)

Nap 6,897,051 dong | Feature CU: 10 | Feature MOI: 13 (+3)
Them moi: ['latest_observed_speed_kmh', 'latest_observed_dur_per_km', 'latest_observed_base_per_km']


**2. Setup & huấn luyện — theo từng tháng, 2 bộ feature song song**

In [2]:
def prep(d, num):
    X = d[CAT + num].copy()
    for cc in CAT: X[cc] = X[cc].astype("category")
    return X

def tao():
    return HistGradientBoostingRegressor(max_iter=500, learning_rate=0.05, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, categorical_features=CAT, random_state=42)

def metrics(y, p):
    y, p = np.asarray(y), np.asarray(p)
    return dict(MAE=mean_absolute_error(y,p), RMSE=mean_squared_error(y,p)**.5,
                R2=r2_score(y,p), MAPE=np.mean(np.abs((y-p)/y))*100)

thangs = sorted(df.evaluation_month.unique())
tests_cu, tests_moi = {}, {}
print("Train theo thang:", thangs)
for th in thangs:
    sub = df[df.evaluation_month==th]
    tr = sub[sub.split=="train"]; te = sub[sub.split=="test"].copy()
    t0 = time.time()

    m_cu = tao().fit(prep(tr, B_NUM_CU), np.log(tr.base_price))
    te["pred_cu"] = np.exp(m_cu.predict(prep(te, B_NUM_CU)))

    m_moi = tao().fit(prep(tr, B_NUM_MOI), np.log(tr.base_price))
    te["pred_moi"] = np.exp(m_moi.predict(prep(te, B_NUM_MOI)))

    tests_cu[th] = te[["evaluation_month","base_price","pred_cu"]]
    tests_moi[th] = te[["evaluation_month","base_price","pred_moi"]]
    print(f"  [{th}] n_train={len(tr):,} n_test={len(te):,} | {time.time()-t0:.1f}s")

Train theo thang: ['2026-01', '2026-02', '2026-03']
  [2026-01] n_train=1,544,286 n_test=315,360 | 38.5s
  [2026-02] n_train=1,547,985 n_test=234,632 | 35.3s
  [2026-03] n_train=1,549,528 n_test=314,368 | 36.6s


**3. So sánh kết quả — CŨ vs MỚI, tổng gộp & từng test-set nhỏ theo tháng**

In [3]:
all_cu = pd.concat(tests_cu.values())
all_moi = pd.concat(tests_moi.values())

rows = []
for ten, d, col in [("CU (10 feature)", all_cu, "pred_cu"), ("MOI (+3 feature)", all_moi, "pred_moi")]:
    rows.append({"Bo feature": ten, "Test-set": "TAT CA", "n": len(d), **metrics(d.base_price, d[col])})
    for th in thangs:
        dt = d[d.evaluation_month==th]
        rows.append({"Bo feature": ten, "Test-set": th, "n": len(dt), **metrics(dt.base_price, dt[col])})
bang = pd.DataFrame(rows).round(2)
print("TONG GOP:"); display(bang[bang["Test-set"]=="TAT CA"])
print("\nTUNG TEST-SET NHO THEO THANG:"); display(bang[bang["Test-set"]!="TAT CA"])

mae_cu = bang[(bang["Test-set"]=="TAT CA")&(bang["Bo feature"]=="CU (10 feature)")]["MAE"].values[0]
mae_moi = bang[(bang["Test-set"]=="TAT CA")&(bang["Bo feature"]=="MOI (+3 feature)")]["MAE"].values[0]
print(f"\nChenh lech MAE (MOI - CU) = {mae_moi-mae_cu:+,.0f} VND")
print("-> Am: feature moi giup ich, nen them vao _common_train.py (B_NUM).")
print("-> Duong/gan 0: feature moi khong giup, giu nguyen bo feature cu.")

TONG GOP:


,Bo feature,Test-set,n,MAE,RMSE,R2,MAPE
0,CU (10 feature),TAT CA,864360,15031.56,20076.67,0.66,14.58
4,MOI (+3 feature),TAT CA,864360,15030.78,20073.62,0.66,14.58



TUNG TEST-SET NHO THEO THANG:


,Bo feature,Test-set,n,MAE,RMSE,R2,MAPE
1,CU (10 feature),2026-01,315360,15066.55,20166.61,0.66,14.60
2,CU (10 feature),2026-02,234632,14968.72,19930.88,0.66,14.56
3,CU (10 feature),2026-03,314368,15043.35,20094.64,0.65,14.59
5,MOI (+3 feature),2026-01,315360,15066.92,20164.01,0.66,14.60
6,MOI (+3 feature),2026-02,234632,14966.35,19924.67,0.66,14.56
7,MOI (+3 feature),2026-03,314368,15042.62,20093.47,0.65,14.59



Chenh lech MAE (MOI - CU) = -1 VND
-> Am: feature moi giup ich, nen them vao _common_train.py (B_NUM).
-> Duong/gan 0: feature moi khong giup, giu nguyen bo feature cu.


**4. Permutation importance của 3 feature mới (trên model MỚI, tháng cuối)**

Xem feature mới có thực sự được model dùng đến hay không (không chỉ nhìn MAE tổng).

In [4]:
from sklearn.inspection import permutation_importance
th_last = thangs[-1]
sub = df[df.evaluation_month==th_last]
tr = sub[sub.split=="train"]; te = sub[sub.split=="test"]
m_last = tao().fit(prep(tr, B_NUM_MOI), np.log(tr.base_price))
r = permutation_importance(m_last, prep(te, B_NUM_MOI), np.log(te.base_price), n_repeats=5, random_state=42, n_jobs=1)
imp = pd.DataFrame({"Feature": CAT+B_NUM_MOI, "Importance": r.importances_mean}).sort_values("Importance", ascending=False)
print(f"Permutation importance (thang {th_last}) - highlight feature moi:")
imp["MOI?"] = imp.Feature.isin(FEATURE_MOI)
display(imp.reset_index(drop=True))

Permutation importance (thang 2026-03) - highlight feature moi:


,Feature,Importance,MOI?
0,quote_distance,0.684374,False
1,quote_duration,0.232937,False
2,service_name,0.012200,False
3,pickup_location_name,0.001983,False
4,history_60m_price_mean,0.000623,False
5,gio_vn,0.000591,False
6,history_60m_price_std,0.000111,False
7,latest_observed_base,0.000078,False
8,latest_observed_quote_distance,0.000067,False
9,history_60m_price_slope_per_minute,0.000043,False


**Kết luận**

- Bước 3: so MAE/MAPE trực tiếp giữa 2 bộ feature trên cùng target `base_price`, cùng test-set —
  câu trả lời dứt khoát có nên thêm 3 feature mới vào `_common_train.py` (`B_NUM`) hay không.
- Bước 4: dù MAE tổng thể có cải thiện nhẹ hay không, permutation importance cho biết model có
  **thực sự dùng** 3 feature mới hay bỏ qua (importance ~0) — bổ sung góc nhìn ngoài MAE.